# Walking the latent space

The 30×30 grid of decoded digits, the concept vectors hiding in it, and the interpolation that pixel space could not do.

**Runs on:** CPU — about 2 minutes · needs the VAE from notebook 01 &nbsp;·&nbsp; **Slides:** [Chapter 17 — Image Generation](../../../course-web-slides/ch17/index.html) &nbsp;·&nbsp; **Section:** 01 — Variational autoencoders

---

## The grid

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

n = 30
digit_size = 28
figure = np.zeros((digit_size * n, digit_size * n))

grid_x = np.linspace(-1, 1, n)
grid_y = np.linspace(-1, 1, n)[::-1]

for i, yi in enumerate(grid_y):
    for j, xi in enumerate(grid_x):
        z_sample = np.array([[xi, yi]])
        x_decoded = vae.decoder.predict(z_sample, verbose=0)
        digit = x_decoded[0].reshape(digit_size, digit_size)
        figure[i * digit_size:(i + 1) * digit_size,
               j * digit_size:(j + 1) * digit_size] = digit

plt.figure(figsize=(13, 13))
plt.imshow(figure, cmap="Greys_r")
plt.xticks([]); plt.yticks([])
plt.xlabel("z[0]"); plt.ylabel("z[1]")
plt.title("900 digits, none of which are in MNIST")
plt.show()

**A completely continuous distribution.** One digit morphs into another as you follow any path. There are no gaps, and that is the KL term's doing.

Specific directions have meaning — a direction for *four-ness*, one for *one-ness*. Nobody labelled them.

## Finding a concept vector

In [ ]:
import keras
from keras.datasets import mnist

(_, y_train), (x_test, y_test) = mnist.load_data()
_, (xt, yt) = mnist.load_data()
xt = np.expand_dims(xt, -1).astype("float32") / 255

z_mean, _ = vae.encoder.predict(xt[:6000], verbose=0)
labels = yt[:6000]

centroids = np.stack([z_mean[labels == d].mean(axis=0) for d in range(10)])
for d in range(10):
    print(f"digit {d}: centroid {centroids[d].round(3)}")

# A "concept vector": the direction from one digit's region to another's.
v = centroids[9] - centroids[4]
print(f"\n4 -> 9 direction: {v.round(3)}")

In [ ]:
start = centroids[4]
steps = np.linspace(0, 1, 9)
imgs = vae.decoder.predict(
    np.stack([start + t * v for t in steps]), verbose=0)

fig, axes = plt.subplots(1, 9, figsize=(13, 1.9))
for ax, im, t in zip(axes, imgs, steps):
    ax.imshow(im[:, :, 0], cmap="gray_r"); ax.axis("off")
    ax.set_title(f"{t:.2f}", fontsize=8)
plt.suptitle("Walking the 4 -> 9 concept vector", y=1.12)
plt.show()

The same *word arithmetic* idea as chapter 15's `V(king) − V(man) + V(woman)`, in pixels. **A direction in the space corresponds to a meaningful change**, and this is what chapter 17's slides mean by the space being *suitable to manipulation via concept vectors*.

## The comparison that matters: pixel space against latent space

In [ ]:
a_idx = np.where(yt == 4)[0][0]
b_idx = np.where(yt == 9)[0][0]
a, b = xt[a_idx], xt[b_idx]

# Pixel-space interpolation, from chapter 5's notebook 02.
alphas = np.linspace(0, 1, 9)
pixel = [(1 - t) * a + t * b for t in alphas]

# Latent-space interpolation.
za, _ = vae.encoder.predict(a[None], verbose=0)
zb, _ = vae.encoder.predict(b[None], verbose=0)
latent = vae.decoder.predict(
    np.stack([(1 - t) * za[0] + t * zb[0] for t in alphas]), verbose=0)

fig, axes = plt.subplots(2, 9, figsize=(13, 3.4))
for j, t in enumerate(alphas):
    axes[0, j].imshow(pixel[j][:, :, 0], cmap="gray_r"); axes[0, j].axis("off")
    axes[1, j].imshow(latent[j][:, :, 0], cmap="gray_r"); axes[1, j].axis("off")
axes[0, 0].set_title("pixel space", loc="left", fontsize=10)
axes[1, 0].set_title("latent space", loc="left", fontsize=10)
plt.tight_layout(); plt.show()

**The top row is ghosts** — two digits superimposed, not a digit. The bottom row is valid digits all the way across.

Chapter 5 showed the top row and promised the bottom one. This is the payoff, and it is the single clearest picture of what *representation learning* buys.

## Where the classes actually sit

In [ ]:
plt.figure(figsize=(8, 7))
sc = plt.scatter(z_mean[:, 0], z_mean[:, 1], c=labels, cmap="tab10",
                 s=5, alpha=.6)
for d in range(10):
    plt.annotate(str(d), centroids[d], fontsize=16, weight="bold",
                 ha="center", va="center",
                 bbox=dict(boxstyle="circle", fc="w", alpha=.8))
plt.colorbar(sc); plt.title("The latent space, labelled")
plt.show()

Confusable digits are adjacent — 4, 9 and 7 share a region; 3, 5 and 8 share another. **The geometry predicts the confusions**, the same relationship chapter 10's notebook 04 found in a classifier's penultimate layer.

## The limitation of two dimensions

In [ ]:
print("A 2-d latent space is plottable, which is why it was chosen.")
print("It is also very tight: 784 pixels compressed to 2 numbers.")
print()
print("Try latent_dim = 8 or 32 in notebook 01 and compare:")
print("  - reconstruction quality will improve noticeably")
print("  - the grid visualization stops being possible")
print("  - t-SNE (chapter 10) becomes the way to look at it")
print()
print("Chapter 17's diffusion models work in a latent space of 16")
print("channels at 1/8 resolution -- thousands of dimensions.")

---

## What to take away

- The decoded grid is continuous everywhere — that is the KL term's effect, visible.
- Directions in the space are **concept vectors**, the pixel analogue of chapter 15's word arithmetic.
- Latent interpolation gives valid digits where pixel interpolation gives ghosts.
- Two dimensions are plottable and tight; real models use thousands.